<a href="https://colab.research.google.com/github/srihari-g-acharya-ci109/ML2LAB/blob/main/ML6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# loan approval dataset (14 instances)
loan_records = [
    {"Job": "Salaried",   "Credit": "Good", "Collateral": "Yes", "PriorDefault": "No",  "Approved": "Yes"},
    {"Job": "Salaried",   "Credit": "Good", "Collateral": "No",  "PriorDefault": "No",  "Approved": "Yes"},
    {"Job": "Business",   "Credit": "Good", "Collateral": "Yes", "PriorDefault": "No",  "Approved": "Yes"},
    {"Job": "Business",   "Credit": "Fair", "Collateral": "Yes", "PriorDefault": "No",  "Approved": "Yes"},
    {"Job": "Unemployed", "Credit": "Fair", "Collateral": "No",  "PriorDefault": "No",  "Approved": "No"},
    {"Job": "Unemployed", "Credit": "Poor", "Collateral": "No",  "PriorDefault": "Yes", "Approved": "No"},
    {"Job": "Salaried",   "Credit": "Poor", "Collateral": "No",  "PriorDefault": "Yes", "Approved": "No"},
    {"Job": "Business",   "Credit": "Poor", "Collateral": "Yes", "PriorDefault": "Yes", "Approved": "No"},
    {"Job": "Salaried",   "Credit": "Fair", "Collateral": "Yes", "PriorDefault": "No",  "Approved": "Yes"},
    {"Job": "Unemployed", "Credit": "Good", "Collateral": "Yes", "PriorDefault": "No",  "Approved": "Yes"},
    {"Job": "Salaried",   "Credit": "Fair", "Collateral": "No",  "PriorDefault": "Yes", "Approved": "No"},
    {"Job": "Business",   "Credit": "Fair", "Collateral": "No",  "PriorDefault": "No",  "Approved": "Yes"},
    {"Job": "Business",   "Credit": "Good", "Collateral": "No",  "PriorDefault": "Yes", "Approved": "No"},
    {"Job": "Salaried",   "Credit": "Poor", "Collateral": "Yes", "PriorDefault": "No",  "Approved": "Yes"},
]

df = pd.DataFrame(loan_records)
features = ["Job", "Credit", "Collateral", "PriorDefault"]
target = "Approved"
pos_label = "Yes"

print("Dataset:")
display(df)
print("
Class distribution:
", df[target].value_counts())

In [ ]:
# helper functions
def rule_covers(rule, row):
    return all(row[attr] == val for attr, val in rule.items())

def eval_rule(rule, data):
    subset = data[data.apply(lambda r: rule_covers(rule, r), axis=1)]
    p = (subset[target] == pos_label).sum()
    n = (subset[target] != pos_label).sum()
    prec = p / (p + n) if (p + n) > 0 else 0.0
    return p, n, prec

def rule_str(rule):
    if not rule:
        return "IF (true) THEN Approved=Yes"
    conds = " AND ".join(f"{k}={v}" for k, v in rule.items())
    return f"IF {conds} THEN Approved=Yes"

# sequential covering with beam search
def learn_one_rule_beam(data, beam_size=3):
    beam = [{}]
    best_rule = {}
    best_prec = 0.0
    
    while True:
        candidates = []
        for cand in beam:
            for f in features:
                if f in cand:
                    continue
                for val in data[f].unique():
                    new_r = {**cand, f: val}
                    p, n, prec = eval_rule(new_r, data)
                    if p > 0:
                        candidates.append((new_r, prec, p))
        if not candidates:
            break
        candidates.sort(key=lambda x: (x[1], x[2]), reverse=True)
        beam = [c[0] for c in candidates[:beam_size]]
        top_rule, top_prec, top_p = candidates[0]
        
        if top_prec > best_prec:
            best_rule = top_rule
            best_prec = top_prec
        else:
            break
            
    return best_rule

def sequential_covering(data, min_prec=0.85):
    remaining = data.copy()
    rules = []
    sc_logs = []
    
    while (remaining[target] == pos_label).any():
        rule = learn_one_rule_beam(remaining)
        p, n, prec = eval_rule(rule, remaining)
        if prec < min_prec or p == 0:
            break
        rules.append(rule)
        sc_logs.append({"rule": rule_str(rule), "pos": p, "neg": n, "prec": prec})
        print(f"Learned: {rule_str(rule)} [prec={prec:.2f}, pos={p}, neg={n}]")
        
        # eliminate covered positives
        mask = remaining.apply(lambda r: rule_covers(rule, r), axis=1)
        remaining = remaining[~(mask & (remaining[target] == pos_label))].reset_index(drop=True)
        
    return rules, sc_logs

print("Running Sequential Covering:")
sc_rules, sc_logs = sequential_covering(df)

In [ ]:
# AQ Algorithm: generates a STAR for a seed positive vs negatives
def build_star(seed, negatives):
    star = [{}]
    for _, neg_row in negatives.iterrows():
        new_star = []
        for rule in star:
            if not rule_covers(rule, neg_row):
                new_star.append(rule)
                continue
            for f in features:
                if f in rule:
                    continue
                seed_v = seed[f]
                neg_v = neg_row[f]
                if seed_v != neg_v:
                    specialized = {**rule, f: seed_v}
                    if rule_covers(specialized, seed):
                        new_star.append(specialized)
        # remove duplicate conditions
        unique_star = []
        for r in new_star:
            if r not in unique_star:
                unique_star.append(r)
        star = unique_star
    return star

def run_aq(data):
    remaining = data.copy()
    aq_rules = []
    aq_logs = []
    
    while (remaining[target] == pos_label).any():
        positives = remaining[remaining[target] == pos_label]
        negatives = remaining[remaining[target] != pos_label]
        
        # select first uncovered positive as seed
        seed = positives.iloc[0]
        star = build_star(seed, negatives)
        
        if not star:
            break
            
        # pick best rule based on precision and brevity
        best_r = max(star, key=lambda r: eval_rule(r, remaining)[2] - 0.05 * len(r))
        p, n, prec = eval_rule(best_r, remaining)
        
        aq_rules.append(best_r)
        aq_logs.append({"rule": rule_str(best_r), "pos": p, "neg": n, "prec": prec})
        print(f"AQ Rule: {rule_str(best_r)} [prec={prec:.2f}, pos={p}, neg={n}]")
        
        mask = remaining.apply(lambda r: rule_covers(best_r, r), axis=1)
        remaining = remaining[~(mask & (remaining[target] == pos_label))].reset_index(drop=True)
        
    return aq_rules, aq_logs

print("
Running AQ Algorithm:")
aq_rules, aq_logs = run_aq(df)

In [ ]:
# plot coverage comparison for sequential covering rules
if sc_logs:
    labels = [f"Rule {i+1}" for i in range(len(sc_logs))]
    pos_cov = [l["pos"] for l in sc_logs]
    neg_cov = [l["neg"] for l in sc_logs]
    precs = [l["prec"] for l in sc_logs]
    
    fig, ax1 = plt.subplots(figsize=(8, 4))
    x = range(len(labels))
    w = 0.35
    
    ax1.bar([i - w/2 for i in x], pos_cov, w, label="Positives Covered", color="#2ca02c")
    ax1.bar([i + w/2 for i in x], neg_cov, w, label="Negatives Covered", color="#d62728")
    ax1.set_ylabel("Count")
    ax1.set_xticks(list(x))
    ax1.set_xticklabels(labels)
    ax1.set_title("Sequential Covering — Coverage per Rule")
    ax1.legend(loc="upper left")
    
    ax2 = ax1.twinx()
    ax2.plot(list(x), precs, color="navy", marker="o", linewidth=2, label="Precision")
    ax2.set_ylabel("Precision")
    ax2.set_ylim(0, 1.1)
    ax2.legend(loc="upper right")
    
    plt.tight_layout()
    plt.show()